In [1]:
pip install opencv-python ultralytics pyttsx3

In [ ]:
import cv2
import pyttsx3
from ultralytics import YOLO

# Initialize YOLOv8 model
model = YOLO("yolov8n.pt")  # Try yolov8s.pt for better accuracy

# Object width reference (in cm) for distance estimation
KNOWN_WIDTHS = {
    "person": 45, "bottle": 7, "chair": 50, "mobile phone": 7,
    "car": 180, "bus": 250, "truck": 250
}

FOCAL_LENGTH = 700  # Approximate focal length for estimation

# Initialize text-to-speech engine
engine = pyttsx3.init()

# Function to estimate distance
def estimate_distance(known_width, perceived_width):
    if perceived_width > 0:
        return (known_width * FOCAL_LENGTH) / perceived_width
    return None

# Function to convert distance to speech format
def format_distance(distance):
    if distance < 100:  # If distance is less than 1 meter (100 cm), use cm
        return f"{int(distance)} centimeters"
    else:  # If distance is 1 meter or more, convert to meters
        return f"{distance / 100:.1f} meters"

# Function to announce detected objects
def speak_objects(objects):
    if objects:
        message = ", ".join(objects)
        engine.say(message)
        engine.runAndWait()

# Open the webcam
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Perform object detection
    results = model.predict(frame, verbose=False)
    detected_items = []

    for box in results[0].boxes:
        class_id = int(box.cls[0])
        label = model.names[class_id]
        x_min, y_min, x_max, y_max = map(int, box.xyxy[0])
        box_width = x_max - x_min

        # Calculate distance if object is in the known list
        if label in KNOWN_WIDTHS:
            distance = estimate_distance(KNOWN_WIDTHS[label], box_width)
            if distance:
                formatted_distance = format_distance(distance)
                detected_items.append(f"{label} is {formatted_distance} away")

                # Draw bounding box and distance label
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                cv2.putText(frame, f"{label}: {formatted_distance}", 
                            (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                            0.5, (0, 255, 0), 2)

    # Speak detected objects
    speak_objects(detected_items)

    # Show the frame
    cv2.imshow("Object Detection", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()